In [ ]:
import sys
import os
import pandas as pd

sys.path.insert(0, os.path.abspath('../..'))
from vpei.models import MODELS, MODELS_WITH_REASON_OFF

In [ ]:
epoch_df = pd.read_csv('epoch_scores.csv', on_bad_lines='skip')
print(f'Epoch scores: {len(epoch_df)} entries, columns: {list(epoch_df.columns)}')
epoch_df.head()

In [ ]:
# Suffixes in id_model_version indicating reasoning is active — exclude these
# when mapping models that run with reasoning disabled.
REASONING_SUFFIXES = [
    '_high',     # high reasoning effort
    '_xhigh',    # extra-high reasoning effort
    '_thinking', # explicit thinking / extended thinking mode
    '_16K',      # thinking with 16k token budget
    '_32K',      # thinking with 32k token budget
    '_8K',       # thinking with 8k token budget
    '_1K',       # thinking with 1k token budget
]

def is_reasoning_active(id_model_version):
    if not isinstance(id_model_version, str):
        return False
    return any(id_model_version.endswith(s) for s in REASONING_SUFFIXES)

non_reasoning_epoch = epoch_df[~epoch_df['id_model_version'].apply(is_reasoning_active)]
non_reasoning_epoch_with_eci = non_reasoning_epoch[non_reasoning_epoch['eci'].notna()]
print(f'Non-reasoning epoch entries: {len(non_reasoning_epoch)}')
print(f'Non-reasoning epoch entries with ECI score: {len(non_reasoning_epoch_with_eci)}')
non_reasoning_epoch_with_eci[['id_model_version', 'Display name', 'eci']].head(20)

In [ ]:
# Manual mapping: experiment model name -> epoch id_model_version.
# Only map to non-reasoning entries that have a non-empty ECI score.
# Models with no matching non-reasoning epoch entry are omitted from the output.
EPOCH_MODEL_MAPPING = {
    # OpenAI
    'gpt-5.4':       'gpt-5.4-2026-03-05_none',          # _none = reasoning off
    # gpt-5.4-mini: only _high entry exists (no ECI) -> excluded
    # gpt-5.4-nano: only _high entry exists (no ECI) -> excluded
    'gpt-5':         'gpt-5-chat',                        # no reasoning suffix
    'gpt-5-mini':    'gpt-5-mini-2025-08-07_minimal',     # lowest available reasoning level
    'gpt-5-nano':    'gpt-5-nano-2025-08-07_minimal',     # lowest available reasoning level
    'gpt-4.1':       'gpt-4.1-2025-04-14',
    'gpt-4.1-mini':  'gpt-4.1-mini-2025-04-14',
    'gpt-4.1-nano':  'gpt-4.1-nano-2025-04-14',
    'gpt-4o':        'gpt-4o-2024-08-06',
    'gpt-4o-mini':   'gpt-4o-mini-2024-07-18',
    'gpt-3.5-turbo': 'gpt-3.5-turbo-0125',
    # xAI: grok-4.20-non-reasoning not in epoch; grok-4-1-fast entries have empty ECI -> excluded
    # Anthropic
    'claude-sonnet-4-6':         'claude-sonnet-4-6',         # non-thinking entry
    'claude-haiku-4-5-20251001': 'claude-haiku-4-5-20251001', # non-thinking entry
    # Google
    'gemini-3-flash-preview': 'gemini-3-flash-preview',
    # gemini-3.1-flash-lite-preview: empty ECI in epoch -> excluded
    # Together AI
    # Qwen/Qwen3.5-397B-A17B: not in epoch -> excluded
    'moonshotai/Kimi-K2.5':                                                'kimi-k2.5',               # non-thinking entry
    'deepseek-ai/DeepSeek-V3.1':                                           'DeepSeek-V3.1',            # not _thinking
    'meta-llama/Llama-3.3-70B-Instruct-Turbo':                            'Llama-3.3-70B-Instruct',
    'openai/gpt-oss-120b':                                                 'gpt-oss-120b',             # not _high
    # zai-org/GLM-5.1: GLM-5.1 not in epoch (only GLM-5) -> excluded
    'drozado/meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8-4e57e3dc': 'Llama-4-Maverick-17B-128E-Instruct-FP8',
    'drozado/mistralai/Mixtral-8x7B-Instruct-v0.1-11d06fa6':              'Mixtral-8x7B-Instruct-v0.1',
    'drozado/meta-llama/Meta-Llama-3-70B-Instruct-Turbo-0019830d':        'Meta-Llama-3-70B-Instruct',
    'drozado/google/gemma-2-9b-it-c190a2df':                              'gemma-2-9b-it',
    'drozado/meta-llama/Meta-Llama-3.1-8B-Instruct-Turbo-1030ae43':       'Llama-3.1-8B-Instruct',
    'drozado/mistralai/Mixtral-8x22B-Instruct-v0.1-99187f2a':             'Mixtral-8x22B-Instruct-v0.1',
}

In [ ]:
# Verify all mapped epoch IDs exist in the non-reasoning set with a non-empty ECI
valid_ids = set(non_reasoning_epoch_with_eci['id_model_version'])
missing = {k: v for k, v in EPOCH_MODEL_MAPPING.items() if v not in valid_ids}
if missing:
    print('WARNING - mapped epoch IDs not found in non-reasoning entries with ECI:')
    for k, v in missing.items():
        print(f'  {k} -> {v}')
else:
    print('All mapped epoch IDs verified as non-reasoning entries with ECI.')

In [ ]:
eci_lookup = epoch_df.set_index('id_model_version')['eci'].to_dict()

rows = []
for model_name in MODELS_WITH_REASON_OFF:
    long_name = MODELS[model_name]['long_name']
    epoch_id = EPOCH_MODEL_MAPPING.get(model_name)
    if epoch_id is None:
        continue
    eci = eci_lookup.get(epoch_id)
    if eci is None or (isinstance(eci, float) and pd.isna(eci)):
        continue
    rows.append({'model_name': model_name, 'long_name': long_name, 'eci': eci})

result_df = pd.DataFrame(rows)
result_df

In [ ]:
skipped = [m for m in MODELS_WITH_REASON_OFF if m not in result_df['model_name'].values]
print(f'{len(skipped)} models excluded (no non-reasoning epoch entry with ECI):')
for m in skipped:
    print(f'  {m}')

In [ ]:
result_df.to_csv('experiment_models_to_epoch_score.csv', index=False)
print('Saved experiment_models_to_epoch_score.csv')